# !!! Notes amélioration données !!!
- Enlever les outliers centre médicaux
- Réintégrer le ENERGYSTARScore
- Normaliser la donnée
- transformer la donnée
- Modififer le type d'encodage
- train test split 'paramétré'

# Analyse Exploratoire

## Import des modules

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

pd.set_option('display.max_columns',None)

## Analyse Exploratoire

In [ ]:
building_consumption = pd.read_csv('2016_Building_Energy_Benchmarking.csv')

In [ ]:
# On regarde comment un batiment est défini dans ce jeu de données 
building_consumption.head()

In [ ]:
# On regarde le nombre de valeurs manquantes par colonne ainsi que leur type 
building_consumption.info()

In [ ]:
building_consumption.columns

A réaliser : 
- Une analyse descriptive des données, y compris une explication du sens des colonnes gardées, des arguments derrière la suppression de lignes ou de colonnes, des statistiques descriptives et des visualisations pertinentes.

Qelques pistes d'analyse : 

* Identifier les colonnes avec une majorité de valeurs manquantes ou constantes en utilisant la méthode value_counts() de Pandas
* Mettre en evidence les différences entre les immeubles mono et multi-usages
* Utiliser des pairplots et des boxplots pour faire ressortir les outliers ou des batiments avec des valeurs peu cohérentes d'un point de vue métier 

Pour vous inspirer, ou comprendre l'esprit recherché dans une analyse exploratoire, vous pouvez consulter ce notebook en ligne : https://www.kaggle.com/code/pmarcelino/comprehensive-data-exploration-with-python. Il ne s'agit pas d'un modèle à suivre à la lettre ni d'un template d'analyses attendues pour ce projet. 

In [ ]:
df = building_consumption.copy()

## Suppression lignes

### Suppresion des lignes residentielles

In [ ]:
dict_BuildingType = {
    'NonResidential' : 'Non-Residential',
    'Multifamily LR (1-4)' :'Residential',
    'Multifamily MR (5-9)' :'Residential',
    'Multifamily HR (10+)': 'Residential',
    'SPS-District K-12' : 'Non-Residential',
    'Nonresidential COS' :'Non-Residential',
    'Campus' :'Non-Residential',
    'Nonresidential WA' :'Non-Residential'
}
df['BuildingType']=df['BuildingType'].map(dict_BuildingType)

df = df.loc[df['BuildingType'] == 'Non-Residential',:]
df.shape

In [ ]:
df = df.loc[(df['PrimaryPropertyType']!='Low-Rise Multifamily') & (df['PrimaryPropertyType']!='Residence Hall'),:] # Résidence universitaire
df.shape

### Suppression d'ENERGYSTARScore temporaire

In [ ]:
df.drop('ENERGYSTARScore',axis=1, inplace=True)

### Suppression de lignes outliers == something

In [ ]:
df = df.loc[df['Outlier'].isna(),:]
df.shape

### Suppression des lignes non compliant

In [ ]:
df = df.loc[df['ComplianceStatus']=='Compliant',:]
df.shape

## Suppression colonnes inutiles

In [ ]:
First_column_to_drop = ['OSEBuildingID', 'DataYear','DefaultData', 'Comments', 'ComplianceStatus','Outlier',"BuildingType",'City','State','TaxParcelIdentificationNumber','ListOfAllPropertyUseTypes','NaturalGas(therms)','Electricity(kWh)','YearsENERGYSTARCertified','PrimaryPropertyType','Address','ZipCode','SiteEUIWN(kBtu/sf)','SourceEUIWN(kBtu/sf)','SiteEnergyUseWN(kBtu)','PropertyName']
df.drop(First_column_to_drop,axis=1, inplace=True)
df.shape

In [ ]:
df = df.loc[df['LargestPropertyUseType'].notna(),:]

## Nettoyage valeurs renseignées

### Nettoyage de primary, secondary et thrid use type

#### mapping

In [ ]:
#Dictionnaire map catégories use type

map_dict_use_type = {
    # -------- 1. Office --------
    "Office": "Office",
    "Financial Office": "Office",
    "Medical Office": "Office",
    "Bank Branch": "Office",

    # -------- 2. Retail --------
    "Retail Store": "Retail",
    "Supermarket/Grocery Store": "Retail",
    "Convenience Store without Gas Station": "Retail",
    "Food Sales": "Retail",
    "Wholesale Club/Supercenter": "Retail",
    "Automobile Dealership": "Retail",
    "Strip Mall": "Retail",
    "Enclosed Mall": "Retail",
    "Lifestyle Center": "Retail",

    # -------- 3. Restaurant & Food --------
    "Restaurant": "Restaurant/Food",
    "Fast Food Restaurant": "Restaurant/Food",
    "Food Service": "Restaurant/Food",
    "Other - Restaurant/Bar": "Restaurant/Food",
    "Bar/Nightclub": "Restaurant/Food",

    # -------- 4. Parking --------
    "Parking": "Parking",

    # -------- 5. Warehouse & Storage --------
    "Non-Refrigerated Warehouse": "Warehouse",
    "Refrigerated Warehouse": "Warehouse",
    "Distribution Center": "Warehouse",
    "Self-Storage Facility": "Warehouse",

    # -------- 6. Housing --------
    "Multifamily Housing": "Housing",
    "Residence Hall/Dormitory": "Housing",
    "Other - Lodging/Residential": "Housing",
    "Senior Care Community": "Housing",
    "Residential Care Facility": "Housing",

    # -------- 7. Hotel --------
    "Hotel": "Hotel",

    # -------- 8. Education --------
    "K-12 School": "Education",
    "Pre-school/Daycare": "Education",
    "College/University": "Education",
    "Vocational School": "Education",
    "Adult Education": "Education",
    "Other - Education": "Education",

    # -------- 9. Healthcare --------
    "Hospital (General Medical & Surgical)": "Healthcare",
    "Other/Specialty Hospital": "Healthcare",
    "Urgent Care/Clinic/Other Outpatient": "Healthcare",

    # -------- 10. Fitness --------
    "Fitness Center/Health Club/Gym": "Fitness",
    "Swimming Pool": "Fitness",

    # -------- 11. Recreation --------
    "Other - Recreation": "Recreation",
    "Social/Meeting Hall": "Recreation",
    "Movie Theater": "Recreation",
    "Performing Arts": "Recreation",

    # -------- 12. Museum/Library --------
    "Museum": "Museum/Library",
    "Library": "Museum/Library",

    # -------- 13. Public Services --------
    "Courthouse": "Public Services",
    "Police Station": "Public Services",
    "Fire Station": "Public Services",
    "Other - Public Services": "Public Services",

    # -------- 14. Science & Technology --------
    "Laboratory": "Science/Technology",
    "Data Center": "Science/Technology",
    "Other - Technology/Science": "Science/Technology",

    # -------- 15. Manufacturing --------
    "Manufacturing/Industrial Plant": "Manufacturing",

    # -------- 16. Religion --------
    "Worship Facility": "Religion",

    # -------- 17. Entertainment --------
    "Other - Entertainment/Public Assembly": "Entertainment",
    "Other - Services": "Entertainment",
    "Other - Mall": "Entertainment",

    # -------- 18. Personal Services --------
    "Personal Services (Health/Beauty, Dry Cleaning, etc)": "Personal Services",
    "Repair Services (Vehicle, Shoe, Locksmith, etc)": "Personal Services",

    # -------- 19. Prison --------
    "Prison/Incarceration": "Prison",

    # -------- 20. Specialty/Other --------
    "Other": "Specialty/Other",
    "Other - Utility": "Specialty/Other"
}


In [ ]:
df['LargestPropertyUseType'] = df['LargestPropertyUseType'].map(map_dict_use_type)
df['SecondLargestPropertyUseType'] = df['SecondLargestPropertyUseType'].map(map_dict_use_type)
df['ThirdLargestPropertyUseType'] = df['ThirdLargestPropertyUseType'].map(map_dict_use_type)

#### remplissage na

In [ ]:
df['SecondLargestPropertyUseTypeGFA'].fillna(0,inplace=True)
df['ThirdLargestPropertyUseTypeGFA'].fillna(0,inplace=True)

### Nettoyage de neighborhood

In [ ]:
df['Neighborhood'] = df['Neighborhood'].str.upper()
df.loc[df['Neighborhood']=="DELRIDGE NEIGHBORHOODS",'Neighborhood'] = "DELRIDGE"
df['Neighborhood'].value_counts()
df.shape

### Nettoyage de number of building

In [ ]:
df[df['NumberofBuildings']==0] = 1

## Analyse des valeurs cibles + suppressions outliers

### PLOT : Distribution GHGEmissionsIntensity

In [ ]:
plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
sns.histplot(x=df['GHGEmissionsIntensity'])
plt.subplot(1,2,2)
sns.boxplot(x=df['GHGEmissionsIntensity'])

In [ ]:
df.drop(index = df[df['GHGEmissionsIntensity']>10].index,inplace=True)
df.shape

### PLOT : Distribution TotalGHGEmissions

In [ ]:
plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
sns.histplot(x=df['TotalGHGEmissions'])
plt.subplot(1,2,2)
sns.boxplot(x=df['TotalGHGEmissions'])

In [ ]:
df.drop(index = df[df['TotalGHGEmissions']>1500].index,inplace=True)
df.shape

ne sont concernés que les centres hospitaliés

### PLOT : Distribution PropertyGFABuilding

In [ ]:
plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
sns.histplot(x=df['PropertyGFABuilding(s)'])
plt.subplot(1,2,2)
sns.boxplot(x=df['PropertyGFABuilding(s)'])

In [ ]:
df.drop(index = df[df['PropertyGFABuilding(s)']>400000].index,inplace=True)
df.shape

### PLOT : Distribution SiteEnergyUse(kBtu) 

In [ ]:
plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
sns.histplot(x=df['SiteEnergyUse(kBtu)'])
plt.subplot(1,2,2)
sns.boxplot(x=df['SiteEnergyUse(kBtu)'])

In [ ]:
df.drop(index = df[df['SiteEnergyUse(kBtu)']>3000000].index,inplace=True)
df.shape

## Analyse des batiments multiusages

In [ ]:
def calcul_multiusage(row):
    if pd.notna(row['ThirdLargestPropertyUseType']):
        return 3
    elif pd.notna(row['SecondLargestPropertyUseType']):
        return 2
    else:
        return 1

df['multiusage'] = df.apply(calcul_multiusage, axis=1)


### PLOT : Number of buildings

In [ ]:
sns.boxplot(data= df, x='NumberofBuildings',y='GHGEmissionsIntensity')

### PLOT : multiusage

In [ ]:
sns.boxplot(data=df,x='multiusage',y='TotalGHGEmissions')

### PLOT : PropertyGFABuilding

In [ ]:

# Définir les bornes des tranches (par exemple jusqu'à 100 000 m²)
bins = range(0, df['PropertyGFABuilding(s)'].max() + 100000, 100000)
df_tranche = df.copy()
# Créer les tranches
df_tranche['tranche_surface'] = pd.cut(df_tranche['PropertyGFABuilding(s)'], bins=bins, right=False)

# Calculer la moyenne des émissions par tranche
resultat = df_tranche.groupby(['tranche_surface','multiusage'])['TotalGHGEmissions'].mean().reset_index(name='em_moyenne')
resultat
sns.barplot(data=resultat,x='tranche_surface',y='em_moyenne',hue='multiusage')

## Analyse des redundances

In [ ]:
Valeurs_numeriques = ['GHGEmissionsIntensity','TotalGHGEmissions','SiteEUI(kBtu/sf)', 'SourceEUI(kBtu/sf)', 'SiteEnergyUse(kBtu)','NumberofBuildings', 'NumberofFloors',
       'PropertyGFATotal', 'PropertyGFAParking', 'PropertyGFABuilding(s)',
        'LargestPropertyUseTypeGFA',
        'SecondLargestPropertyUseTypeGFA',
        'ThirdLargestPropertyUseTypeGFA',
       
       'SteamUse(kBtu)', 'Electricity(kBtu)', 'NaturalGas(kBtu)',
        ]

corrmat = df[Valeurs_numeriques].corr()
plt.figure(figsize=(20,10))
sns.heatmap(corrmat, annot=True, square=True)
plt.xticks(fontsize=6)
plt.yticks(fontsize=6)
plt.show()

In [ ]:
redundance = ['SourceEUI(kBtu/sf)', 'PropertyGFATotal']
df.drop(redundance,axis=1, inplace=True)
df.shape

# Modélisation 

## Import des modules 

In [ ]:
#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error 
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor


## Feature Engineering

### HasEnergies

In [ ]:
df['HasGas'] = df.apply(lambda row : 1 if row['NaturalGas(kBtu)']>0 else 0,axis = 1)
df.shape

In [ ]:
df['HasSteam'] = df.apply(lambda row : 1 if row['SteamUse(kBtu)']>0 else 0,axis = 1)
df.shape

In [ ]:
df['HasElectricity'] = df.apply(lambda row : 1 if row['Electricity(kBtu)']>0 else 0,axis = 1)
df.shape

### Age building 

In [ ]:
df['ageBuilding'] = df.apply(lambda row : 2025 - row['YearBuilt'],axis=1)
df.drop('YearBuilt',axis=1,inplace=True)
df.shape

### Surface moyenne prise en au sol / par étage

In [ ]:
df['sf_floor'] = df.apply(lambda row : row['PropertyGFABuilding(s)']/(row['NumberofFloors']+1) ,axis=1)
df.shape

## DATALEAKAGE

### Energies 

In [ ]:
energies_dataleakage = ['SiteEnergyUse(kBtu)','SteamUse(kBtu)', 'Electricity(kBtu)', 'NaturalGas(kBtu)',]
df.drop(energies_dataleakage,axis=1,inplace=True)
df.shape

## Valeurs cibles

In [ ]:
#'GHGEmissionsIntensity', 'SiteEUI(kBtu/sf)'
df.drop('TotalGHGEmissions',axis=1,inplace=True)
df.shape


In [ ]:
df.head()

## Encodage

In [ ]:
df = pd.get_dummies(df,columns=['Neighborhood'])
df = pd.get_dummies(df,columns=['LargestPropertyUseType'])
df = pd.get_dummies(df,columns=['SecondLargestPropertyUseType'])
df = pd.get_dummies(df,columns=['ThirdLargestPropertyUseType'])
df.shape

## Cible SiteEUI(kBtu/sf)

### Séparation X_energy, y_energy

In [ ]:
y_energy = df['SiteEUI(kBtu/sf)'].copy()
X = df.copy().drop(['SiteEUI(kBtu/sf)','GHGEmissionsIntensity'], axis=1)

### Train test split

In [ ]:
X_train, X_test, y_energy_train, y_energy_test = train_test_split(X, y_energy, random_state=42)

### Standardisation

In [ ]:
X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train =pd.DataFrame(
    X_scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index)


X_test = pd.DataFrame(
    X_scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index)


y_energy_train=pd.DataFrame(
    y_scaler.fit_transform(y_energy_train.to_numpy().reshape(-1, 1)),
    columns=['SiteEUI(kBtu/sf)'],
    index=y_energy_train.index)


y_energy_test=pd.DataFrame(
    y_scaler.transform(y_energy_test.to_numpy().reshape(-1, 1)),
    columns=['SiteEUI(kBtu/sf)'],
    index=y_energy_test.index)

### Models

In [ ]:
df_total = pd.DataFrame(columns=['nom_model','r2', 'MAE','MSE'])

#### DummyRegressor

In [ ]:
dummy_model = DummyRegressor(strategy='mean')
cv = cross_validate(dummy_model,X_train,y_energy_train,cv=5,verbose=1,n_jobs=-1,scoring=['r2','neg_mean_absolute_error','neg_mean_squared_error'])
df_cv = pd.DataFrame(cv)
df_total =pd.concat([df_total, pd.DataFrame({'nom_model': ['dummy_model'],'r2':[df_cv['test_r2'].mean()],'MAE':[df_cv['test_neg_mean_absolute_error'].mean()],'MSE':[df_cv['test_neg_mean_squared_error'].mean()]})])


#### RidgeRegressor

In [ ]:
Ridge_model = Ridge(random_state=42)
cv = cross_validate(Ridge_model,X_train,y_energy_train,cv=5,verbose=1,n_jobs=-1,scoring=['r2','neg_mean_absolute_error','neg_mean_squared_error'])
df_cv = pd.DataFrame(cv)
df_total =pd.concat([df_total, pd.DataFrame({'nom_model': ['Ridge_model'],'r2':[df_cv['test_r2'].mean()],'MAE':[df_cv['test_neg_mean_absolute_error'].mean()],'MSE':[df_cv['test_neg_mean_squared_error'].mean()]})])

#### RandomForestRegressor

In [ ]:
RandomForest_model = RandomForestRegressor(random_state=42)
cv = cross_validate(RandomForest_model,X_train,y_energy_train,cv=5,verbose=1,n_jobs=-1,scoring=['r2','neg_mean_absolute_error','neg_mean_squared_error'])
df_cv = pd.DataFrame(cv)
df_total =pd.concat([df_total, pd.DataFrame({'nom_model': ['RandomForest_model'],'r2':[df_cv['test_r2'].mean()],'MAE':[df_cv['test_neg_mean_absolute_error'].mean()],'MSE':[df_cv['test_neg_mean_squared_error'].mean()]})])

#### GradientBoostingRegressor

In [ ]:
GradientBoosting_model = GradientBoostingRegressor(random_state=42)
cv = cross_validate(GradientBoosting_model,X_train,y_energy_train,cv=5,verbose=1,n_jobs=-1,scoring=['r2','neg_mean_absolute_error','neg_mean_squared_error'])
df_cv = pd.DataFrame(cv)
df_total =pd.concat([df_total, pd.DataFrame({'nom_model': ['GradientBoosting_model'],'r2':[df_cv['test_r2'].mean()],'MAE':[df_cv['test_neg_mean_absolute_error'].mean()],'MSE':[df_cv['test_neg_mean_squared_error'].mean()]})])

#### Comparaison model energy

In [ ]:
df_total

## Cible GHGEmissionsIntensity

In [ ]:
df_total = df_total[0:0]

### Séparation X_CO2, y_CO2

In [ ]:
y_CO2 = df['GHGEmissionsIntensity'].copy()

### Train test split

In [ ]:
X_train, X_test, y_CO2_train, y_CO2_test = train_test_split(X, y_CO2, random_state=42)

### Standardisation

In [ ]:

y_scaler = StandardScaler()

y_CO2_train=pd.DataFrame(
    y_scaler.fit_transform(y_CO2_train.to_numpy().reshape(-1, 1)),
    columns=['GHGEmissionsIntensity'],
    index=y_CO2_train.index)


y_CO2_test=pd.DataFrame(
    y_scaler.transform(y_CO2_test.to_numpy().reshape(-1, 1)),
    columns=['GHGEmissionsIntensity'],
    index=y_CO2_test.index)

### Models

#### DummyRegressor

In [ ]:
dummy_model = DummyRegressor(strategy='mean')
cv = cross_validate(dummy_model,X_train,y_CO2_train,cv=5,verbose=1,n_jobs=-1,scoring=['r2','neg_mean_absolute_error','neg_mean_squared_error'])
df_cv = pd.DataFrame(cv)
df_total =pd.concat([df_total, pd.DataFrame({'nom_model': ['dummy_model'],'r2':[df_cv['test_r2'].mean()],'MAE':[df_cv['test_neg_mean_absolute_error'].mean()],'MSE':[df_cv['test_neg_mean_squared_error'].mean()]})])

#### RidgeRegressor

In [ ]:
Ridge_model = Ridge(random_state=42)
cv = cross_validate(Ridge_model,X_train,y_CO2_train,cv=5,verbose=1,n_jobs=-1,scoring=['r2','neg_mean_absolute_error','neg_mean_squared_error'])
df_cv = pd.DataFrame(cv)
df_total =pd.concat([df_total, pd.DataFrame({'nom_model': ['Ridge_model'],'r2':[df_cv['test_r2'].mean()],'MAE':[df_cv['test_neg_mean_absolute_error'].mean()],'MSE':[df_cv['test_neg_mean_squared_error'].mean()]})])

#### RandomForestRegressor

In [ ]:
RandomForest_model = RandomForestRegressor(random_state=42)
cv = cross_validate(RandomForest_model,X_train,y_CO2_train,cv=5,verbose=1,n_jobs=-1,scoring=['r2','neg_mean_absolute_error','neg_mean_squared_error'])
df_cv = pd.DataFrame(cv)
df_total =pd.concat([df_total, pd.DataFrame({'nom_model': ['RandomForest_model'],'r2':[df_cv['test_r2'].mean()],'MAE':[df_cv['test_neg_mean_absolute_error'].mean()],'MSE':[df_cv['test_neg_mean_squared_error'].mean()]})])

#### GradientBoostingRegressor

In [ ]:
GradientBoosting_model = GradientBoostingRegressor(random_state=42)
cv = cross_validate(GradientBoosting_model,X_train,y_CO2_train,cv=5,verbose=1,n_jobs=-1,scoring=['r2','neg_mean_absolute_error','neg_mean_squared_error'])
df_cv = pd.DataFrame(cv)
df_total =pd.concat([df_total, pd.DataFrame({'nom_model': ['GradientBoosting_model'],'r2':[df_cv['test_r2'].mean()],'MAE':[df_cv['test_neg_mean_absolute_error'].mean()],'MSE':[df_cv['test_neg_mean_squared_error'].mean()]})])

#### Comparaison model CO2

In [ ]:
df_total